# Query-based detector: RT-DETRv2

COCO-pretrained RT-DETRv2 R50vd (Hugging Face `transformers`) on the adopted split: training setup, history, results, tuning experiments with a seed repeat, how bipartite matching replaces NMS, the queries on a dense frame, validation mAP against training time compared with the CNN detectors, mAP by scene density and object size, failure analysis, and a comparison of all four families.

| # | Section |
| --- | --- |
| 1 | [Setup](#1-setup) |
| 2 | [Training setup](#2-training-setup) |
| 3 | [Training history](#3-training-history) |
| 4 | [Results](#4-results) |
| 5 | [Experiments](#5-experiments) |
| 6 | [Inside the model](#6-inside-the-model) |
| 7 | [Failure analysis](#7-failure-analysis) |
| 8 | [Comparison](#8-comparison) |
| 9 | [Conclusions](#9-conclusions) |

## 1. Setup

Imports, data root from `DATA_ROOT`, the shared plot theme, the annotations of the adopted split, and the rows of `experiments/results.csv`, the source of truth for reported numbers. Section 6.3 loads the baseline's `best.pt`, which exists only on the PC.

In [ ]:
import json
import sys
import tomllib
from pathlib import Path

import numpy as np
import polars as pl
import torch
from decouple import AutoConfig
from IPython.display import display

# make the repository root importable when the kernel starts in notebooks/
PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
# decouple locates .env from the caller file, which is a temp file inside a kernel
config = AutoConfig(search_path=PROJECT_ROOT)

from torchvision.io import ImageReadMode, decode_image

from src.data.annotations import load_annotations, load_split_clips
from src.data.classes import CLASS_NAMES
from src.data.dataset import load_frame_steps, select_frames
from src.evaluate import METRICS_FILE, PREDICTIONS_FILE
from src.evaluation.coco_metrics import evaluate_frame_groups
from src.evaluation.failure_analysis import (
    compute_recall_by_size,
    count_error_categories,
    find_best_f1_threshold,
    match_detections,
    select_error_examples,
)
from src.inference.torchvision_detectors import (
    MODEL_STATE_KEY,
    convert_frames_to_inputs,
    select_device,
)
from src.models.rtdetr import build_rtdetr, predict_query_boxes
from src.training.experiment import CONFIG_FILE
from src.visualization.frames import find_densest_crop, plot_crop_panels, plot_error_examples
from src.visualization.plots import plot_grouped_bars, plot_training_history
from src.visualization.style import GT_COLOR, PREDICTION_COLOR, set_plot_theme
from src.visualization.tables import compare_with_reference, load_run_histories

FAMILY = "rtdetr"
BASELINE = "rtdetr-baseline"
SEED_PAIR = ("rtdetr-lr_scaled", "rtdetr-lr_scaled_repeat-s7")
TUNED_BY_FAMILY = {
    "faster_rcnn": "faster_rcnn-giou_loss",
    "retinanet": "retinanet-baseline",
    "yolo": "yolo-baseline",
    "rtdetr": "rtdetr-baseline",
}

SEED = 42
set_plot_theme()
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)
pl.Config.set_fmt_str_lengths(200)

DATA_ROOT = Path(config("DATA_ROOT"))
TUNED_FASTER_RCNN = "faster_rcnn-giou_loss"
tables = load_annotations(DATA_ROOT, load_split_clips(PROJECT_ROOT / "config" / "data.toml"))
val_images = tables.images.filter(pl.col("split") == "val")
val_objects = tables.objects.filter(pl.col("split") == "val")
val_ignored = tables.ignored.filter(pl.col("split") == "val")
results = pl.read_csv(PROJECT_ROOT / "experiments" / "results.csv")
run_dirs = {row["experiment"]: PROJECT_ROOT / row["run_dir"] for row in results.iter_rows(named=True)}
run_configs = {e: tomllib.loads((d / CONFIG_FILE).read_text()) for e, d in run_dirs.items()}
run_metrics = {e: json.loads((d / METRICS_FILE).read_text()) for e, d in run_dirs.items()}
family_results = results.filter(pl.col("experiment").str.starts_with(f"{FAMILY}-"))
family_results.select("experiment", "run_dir", "seed", "machine", "is_dirty")

Four RT-DETRv2 runs, all on the PC from a clean working tree: the baseline, two one-factor experiments, and a seed repeat of `lr_scaled`.

## 2. Training setup

The baseline protocol shared with the other families (AGENTS.md, Data and Experiments): COCO-pretrained weights, frames resized to a shorter side of 1024 px (`max_size` 1824), train frames at step 5, about 1.5 GPU hours on the PC, no augmentation. The optimizer follows the official RT-DETRv2 recipe: AdamW with a 10 times lower backbone LR, linear warmup over 2,000 steps, then a constant LR. Batch 2 with gradient accumulation 2 gives the effective batch of 4 of the torchvision families.

In [ ]:
baseline_config = run_configs[BASELINE]
baseline_training = run_metrics[BASELINE]["training"]
training_fields = [
    "machine", "precision", "frame_step", "epochs", "batch_size", "grad_accum",
    "effective_batch_size", "gpu_hours", "best_epoch",
]
pl.DataFrame(
    [("config", key, str(value)) for key, value in baseline_config.items() if not isinstance(value, dict)]
    + [
        ("training record", key, str(round(value, 2) if isinstance(value, float) else value))
        for key in training_fields
        for value in [baseline_training[key]]
    ],
    schema=["source", "setting", "value"],
    orient="row",
)

- 7 epochs at batch 2 with gradient accumulation 2 (effective 4) took 1.43 GPU hours in bfloat16; the best monitoring epoch is 2.
- AdamW at lr 1e-4 for the encoder and decoder and 1e-5 for the backbone, the official recipe values at batch 16, not rescaled; 2,000 warmup steps cover about 1.6 epochs.

In [ ]:
pl.DataFrame(
    [
        (table, key, str(value))
        for table in ("queries", "matcher", "loss", "inference")
        for key, value in baseline_config[table].items()
    ],
    schema=["table", "setting", "value"],
    orient="row",
)

All query, matcher, and loss settings are the checkpoint defaults: 300 object queries, 100 denoising queries, matching cost 2 (class) / 5 (L1) / 2 (GIoU), varifocal classification and L1 plus GIoU box losses on every decoder layer. Inference keeps the top 300 (query, class) pairs without NMS.

## 3. Training history

Per-epoch monitoring on val frames at step 5 (520 frames) with the shared evaluator; `best.pt` is the epoch with the highest monitoring mAP. All runs use the baseline's data order except the seed repeat.

In [ ]:
family_run_dirs = {e: run_dirs[e] for e in family_results["experiment"]}
histories = load_run_histories(family_run_dirs)
fig = plot_training_history(
    histories,
    {
        "val/map": "Monitoring mAP@[.5:.95]",
        "val/map50": "Monitoring AP50",
        "val/ap_small": "Monitoring AP_small",
    },
    "RT-DETRv2 monitoring metrics per epoch (val, step 5)",
)

- The baseline reaches 0.3200 after one epoch, peaks at 0.3227 at epoch 2, and declines to 0.2960 at epoch 7; monitoring AP_small falls from 0.145 to 0.106 at epoch 6.
- `lr_scaled` starts lower (0.2442 after epoch 1, when its warmup LR is a quarter of the baseline's), peaks higher (0.3315), and declines less (0.3045 at epoch 7); its seed repeat peaks at 0.3230.
- `denoising_groups` follows the baseline with a later peak (0.3296 at epoch 3).

In [ ]:
fig = plot_training_history(
    histories,
    {"train/loss_vfl": "Varifocal classification loss", "train/loss_bbox": "L1 box loss", "train/loss_giou": "GIoU box loss"},
    "RT-DETRv2 training losses per epoch (all decoder layers, encoder, and denoising)",
)

- All losses fall in every epoch; the baseline and `denoising_groups` curves lie on top of each other.
- Both `lr_scaled` runs keep higher training losses than the baseline but equal or higher val mAP: fitting the training frames less closely does not hurt val, consistent with the overfitting hypothesis of the early peak.

In [ ]:
histories.group_by("experiment", maintain_order=True).agg(
    pl.len().alias("epochs"),
    pl.col("epoch").get(pl.col("val/map").arg_max()).alias("best_epoch"),
    pl.col("val/map").max().round(4).alias("best_monitoring_map"),
    pl.col("val/map").last().round(4).alias("last_monitoring_map"),
    pl.col("epoch_seconds").mean().round(0).alias("seconds_per_epoch"),
    pl.col("gpu_memory_gb").max().round(2).alias("peak_gpu_memory_gb"),
)

- Epochs take 731 to 742 s in three runs; the seed repeat took 813 s per epoch, so its gpu_hours (1.59) is not comparable for speed.
- Peak memory is 7.07 GB, 7.14 GB with 200 denoising queries.

## 4. Results

Rows of `experiments/results.csv`: the shared evaluator on all 2,593 val frames with `best.pt`, `maxDets` 500, ignored regions as `iscrowd=1`. RT-DETRv2 keeps at most 300 detections per frame (its query count). ms/image is measured on the PC at batch 1, including the resize.

In [ ]:
RESULT_COLUMNS = [
    "experiment", "epochs", "effective_batch_size", "gpu_hours", "map", "map50", "map75",
    "ap_small", "ap_medium", "ap_large", "map_max_width_1920", "ap_small_max_width_1920",
    "ms_per_image", "public_lb",
]
family_results.select(RESULT_COLUMNS).with_columns(pl.col("gpu_hours", "ms_per_image").round(2))

- Baseline: 0.3209 mAP, 0.6027 AP50, 0.1334 AP_small at 53.2 ms/image, the best val mAP and the best public LB score (0.2563) of all family baselines.
- `lr_scaled` has the highest val mAP of all runs (0.3305), but see the seed repeat below.

## 5. Experiments

Two tuning experiments, each one change against the baseline, chosen from the rubric's options (separate learning rates, denoising): `lr_scaled` scales both learning rates to the effective batch (4 instead of the recipe's 16), `denoising_groups` doubles the denoising queries so that dense frames get several denoising groups. `lr_scaled_repeat-s7` repeats `lr_scaled` with seed 7.

In [ ]:
COMPARED_METRICS = ["map", "map50", "map75", "ap_small", "ap_medium", "ap_large", "map_max_width_1920"]
display(family_results.select("experiment", "epochs", "notes"))
compare_with_reference(family_results, BASELINE, COMPARED_METRICS).with_columns(
    pl.selectors.starts_with("delta_").round(4)
)

- `lr_scaled` +0.0096 mAP (AP_medium +0.0463), `denoising_groups` +0.0065, and the seed repeat of `lr_scaled` +0.0002.
- In both `lr_scaled` seeds AP50 is higher (+0.0113, +0.0095) and AP_large lower (−0.0104, −0.0160) than in the baseline.

In [ ]:
pl.DataFrame(
    {
        "class_name": list(CLASS_NAMES),
        **{
            e: [round(run_metrics[e]["evaluation"]["per_class_ap"][name], 4) for name in CLASS_NAMES]
            for e in family_results["experiment"]
        },
    }
)

- The seed-42 gains come from bus and truck: `lr_scaled` bus 0.530 and truck 0.279 against 0.443 and 0.245, while five of the other six classes lose 0.006 to 0.014 (tricycle +0.003); `denoising_groups` truck 0.312 and bus 0.465, van −0.040.
- In the seed repeat bus falls back to 0.458 and truck to 0.251.

In [ ]:
first, second = SEED_PAIR
seed_histories = histories.filter(pl.col("experiment").is_in(SEED_PAIR)).pivot(
    on="experiment", index="epoch", values="val/map"
)
seed_difference = (pl.col(first) - pl.col(second)).abs()
pl.DataFrame(
    {
        "seed noise measure": [
            "full val mAP, |seed 42 - seed 7|",
            "full val mAP on frames ≤ 1920 px, |seed 42 - seed 7|",
            "monitoring mAP, mean |seed 42 - seed 7| over epochs",
            "monitoring mAP, max |seed 42 - seed 7| over epochs",
            "bus AP, |seed 42 - seed 7|",
        ],
        "value": [
            round(abs(run_metrics[first]["evaluation"]["map"] - run_metrics[second]["evaluation"]["map"]), 4),
            round(abs(run_metrics[first]["evaluation"]["map_max_width_1920"] - run_metrics[second]["evaluation"]["map_max_width_1920"]), 4),
            round(seed_histories.select(seed_difference.mean()).item(), 4),
            round(seed_histories.select(seed_difference.max()).item(), 4),
            round(abs(run_metrics[first]["evaluation"]["per_class_ap"]["bus"] - run_metrics[second]["evaluation"]["per_class_ap"]["bus"]), 4),
        ],
    }
)

- The two seeds of `lr_scaled` differ by 0.0094 full-val mAP, 0.0109 on frames ≤ 1920 px, and 0.0106 monitoring mAP per epoch on average (at most 0.0229); bus AP alone differs by 0.071.
- This is ten times the Faster R-CNN seed difference (0.0009, `02_two_stage.ipynb` §5). Neither RT-DETR experiment exceeds it, so neither is established.

In [ ]:
# lr_scaled has the highest full-val mAP but its seed repeat falls back to the baseline's level
# (table above), so the baseline stays the tuned model of this family
TUNED = BASELINE
TUNED

The tuned RT-DETRv2 is the baseline: `lr_scaled` has the highest val mAP, but its gain does not repeat with seed 7.

## 6. Inside the model

RT-DETRv2 encodes the multi-scale backbone features, selects the 300 highest-scoring encoder positions as object queries, and refines them through 6 decoder layers; each query predicts one box and a sigmoid score per class. In training, the Hungarian algorithm matches each GT box to exactly one query (cost: focal class cost, L1, and GIoU); unmatched queries learn "no object". Denoising queries, built from noised GT boxes, add a direct reconstruction task.

### 6.1 Bipartite matching and no NMS

With one-to-one matching, a second query on an already matched object is trained as "no object", so the model learns to suppress its own duplicates and needs no NMS: inference keeps the top-scoring (query, class) pairs directly. Anchor-based detectors assign many anchors or proposals to each object and remove the duplicate boxes with NMS afterwards. The table compares the detections each model keeps per frame; duplicates at the best-F1 threshold are counted in section 7.2.

In [ ]:
DENSITY_MODELS = list(TUNED_BY_FAMILY.values())
all_predictions = {e: pl.read_parquet(run_dirs[e] / PREDICTIONS_FILE) for e in DENSITY_MODELS}
pl.concat(
    predictions.group_by("image_id").len().select(
        pl.lit(e).alias("experiment"),
        pl.col("len").min().alias("min_detections_per_frame"),
        pl.col("len").median().alias("median_detections_per_frame"),
        pl.col("len").max().alias("max_detections_per_frame"),
    )
    for e, predictions in all_predictions.items()
)

- RT-DETRv2 outputs exactly 300 detections on every val frame (its query count); the NMS-based models keep between 27 and 500, with medians of 345 to 480.
- Without NMS, the number of outputs is fixed by the queries rather than by how many boxes survive suppression.

### 6.2 Queries and scene density

The 300 object queries bound the number of objects per frame. Denoising groups depend on density too: Hugging Face RT-DETRv2 builds `num_denoising // n` groups for a batch whose densest frame has n GT boxes, at least one. The second table counts groups over random pairs of training frames at step 5 (batch 2), for the baseline's 100 and `denoising_groups`' 200 denoising queries.

In [ ]:
objects_per_frame = val_images.select("image_id").join(
    val_objects.group_by("image_id").len(), on="image_id", how="left"
).fill_null(0)
display(
    objects_per_frame.select(
        pl.len().alias("val_frames"),
        pl.col("len").median().alias("median_objects"),
        pl.col("len").quantile(0.95).alias("p95_objects"),
        pl.col("len").max().alias("max_objects"),
        (pl.col("len") > baseline_config["queries"]["num_queries"]).sum().alias("frames_above_queries"),
    )
)
frame_steps = load_frame_steps(PROJECT_ROOT / "config" / "data.toml")
train_counts = (
    select_frames(tables.images.filter(pl.col("split") == "train"), frame_steps["train_tuning"])
    .select("split", "image_id")
    .join(tables.objects.group_by("split", "image_id").len(), on=["split", "image_id"], how="left")
    .fill_null(0)["len"]
    .to_numpy()
)
rng = np.random.default_rng(SEED)
pair_max = np.concatenate(
    [train_counts[rng.permutation(len(train_counts))[: len(train_counts) // 2 * 2]].reshape(-1, 2).max(axis=1) for _ in range(20)]
)
pair_max = pair_max[pair_max > 0]
pl.DataFrame(
    [
        {
            "num_denoising": n,
            "median_groups": float(np.median(np.maximum(1, n // pair_max))),
            "share_single_group": round(float(np.mean(np.maximum(1, n // pair_max) == 1)), 3),
            "max_denoising_queries": int(np.max(2 * pair_max * np.maximum(1, n // pair_max))),
        }
        for n in (100, 200)
    ]
)

- Val frames have a median of 38 GT boxes, 104 at the 95th percentile, and at most 116: no frame has more objects than the 300 queries.
- With 100 denoising queries, 59% of training batches get a single denoising group (median 1); with 200, 15% (median 3) at no more than 400 denoising queries. Dense scenes therefore weaken the denoising task of the default setting; `denoising_groups` tested this (section 5).

### 6.3 Top-query boxes on a dense frame

The `val` frame with the most GT boxes, cropped to the 960 × 540 px window with the most GT box centers. Left: GT boxes; middle: the boxes of all 300 queries of the baseline's `best.pt`, each with its best class; right: its detections at the best-F1 score threshold of section 7.2. This cell loads `best.pt` and needs the PC.

In [ ]:
device = select_device()
detector = build_rtdetr(baseline_config)
checkpoint = torch.load(run_dirs[BASELINE] / "best.pt", map_location="cpu", weights_only=True)
detector.load_state_dict(checkpoint[MODEL_STATE_KEY])
detector = detector.to(device).eval()
dense_frame = (
    objects_per_frame.join(val_images, on="image_id")
    .sort("len", "image_id", descending=[True, False])
    .row(0, named=True)
)
frame = decode_image(dense_frame["file_path"], mode=ImageReadMode.RGB)
queries = predict_query_boxes(detector, convert_frames_to_inputs([frame], device)[0])
baseline_matches = match_detections(tables, "val", all_predictions[BASELINE])
baseline_threshold = find_best_f1_threshold(baseline_matches)["threshold"]
is_dense_frame = pl.col("image_id") == dense_frame["image_id"]
dense_objects = val_objects.filter(is_dense_frame)
CROP_SIZE = (960, 540)
crop_corner = find_densest_crop(dense_objects, (dense_frame["width"], dense_frame["height"]), CROP_SIZE, float("inf"))
fig = plot_crop_panels(
    Path(dense_frame["file_path"]),
    val_ignored.filter(is_dense_frame),
    crop_corner,
    CROP_SIZE,
    [
        (f"Ground truth, {dense_objects.height} boxes in the frame", dense_objects, GT_COLOR),
        (f"All {queries.height} query boxes", queries, PREDICTION_COLOR),
        (f"Detections with score ≥ {baseline_threshold:.2f}",
         all_predictions[BASELINE].filter(is_dense_frame, pl.col("score") >= baseline_threshold),
         PREDICTION_COLOR),
    ],
    f"{BASELINE} on {dense_frame['clip']} frame {dense_frame['frame_id']}",
)
queries.select(
    pl.len().alias("queries"),
    (pl.col("score") >= baseline_threshold).sum().alias("queries_above_threshold"),
    pl.lit(dense_objects.height).alias("gt_boxes"),
    pl.col("score").quantile(0.5).round(3).alias("median_query_score"),
)

*To be filled after the run on the PC: this output needs the baseline's `best.pt`.*

### 6.4 Validation mAP against training time

Monitoring mAP (val at step 5) against the cumulative training time of each run, for the three families trained with the shared loop: the time includes the per-epoch monitoring, as in the GPU-hour budget.

In [ ]:
time_runs = {e: run_dirs[e] for e in ("faster_rcnn-baseline", TUNED_FASTER_RCNN, "retinanet-baseline", BASELINE)}
timed = load_run_histories(time_runs).with_columns(
    (pl.col("epoch_seconds").cum_sum().over("experiment") / 3600).alias("cumulative_hours")
)
fig = plot_training_history(
    timed,
    {"val/map": "Monitoring mAP@[.5:.95]", "val/ap_small": "Monitoring AP_small"},
    "Monitoring mAP against cumulative training time (PC)",
    x="cumulative_hours",
    xlabel="Cumulative training time (GPU hours)",
)
timed.group_by("experiment", maintain_order=True).agg(
    pl.col("cumulative_hours").first().round(2).alias("hours_epoch_1"),
    pl.col("val/map").first().round(4).alias("map_epoch_1"),
    pl.col("cumulative_hours").get(pl.col("val/map").arg_max()).round(2).alias("hours_to_best"),
    pl.col("val/map").max().round(4).alias("best_map"),
    pl.col("cumulative_hours").last().round(2).alias("total_hours"),
    pl.col("val/map").last().round(4).alias("last_map"),
)

- RT-DETRv2 reaches 0.3200 monitoring mAP after its first epoch (0.20 GPU hours) and its best, 0.3227, after 0.40 hours; Faster R-CNN reaches 0.2981 after 0.14 hours and its best, 0.3024, after 0.27 hours. RetinaNet peaks later (0.2676 after 0.46 hours).
- From its first epoch on, the RT-DETRv2 curve lies above the CNN detectors trained with the same loop, but all of them reach their best within the first third of the 1.5 hour budget and decline afterwards.
- **Convergence statement.** On val, RT-DETRv2 converged within the budget: its monitoring mAP peaks after 0.4 of 1.4 GPU hours. Its training losses have not converged and still fall at epoch 7; the gap between falling training losses and falling val mAP points to overfitting, not to a lack of training time.

### 6.5 mAP by scene density and object size

All val frames grouped by their number of GT boxes, each group evaluated separately with the shared evaluator (`src.evaluation.coco_metrics.evaluate_frame_groups`), for the tuned model of every family. AP by object size over all frames is in the results rows (section 8).

In [ ]:
density_groups = objects_per_frame.rename({"len": "objects_in_frame"}).with_columns(
    (pl.when(pl.col("objects_in_frame") < 25).then(pl.lit("1 to 24"))
.when(pl.col("objects_in_frame") < 50).then(pl.lit("25 to 49"))
.when(pl.col("objects_in_frame") < 100).then(pl.lit("50 to 99"))
.otherwise(pl.lit("100 or more"))).alias("group")
).select("image_id", "group")
by_density = pl.concat(
    evaluate_frame_groups(tables, "val", all_predictions[e], density_groups).with_columns(pl.lit(e).alias("experiment"))
    for e in DENSITY_MODELS
)
ORDER = pl.Enum(["1 to 24", "25 to 49", "50 to 99", "100 or more"])
display(by_density.select("group", "frames", "objects").unique(maintain_order=True).sort(pl.col("group").cast(ORDER)))
fig = plot_grouped_bars(
    by_density.sort(pl.col("group").cast(ORDER), maintain_order=True),
    "group",
    "map",
    "experiment",
    ("GT boxes in the frame", "mAP@[.5:.95]", "mAP by scene density (val)"),
    "{:.3f}",
)
by_density.pivot(on="experiment", index="group", values=["map", "ap_small"]).sort(pl.col("group").cast(ORDER)).with_columns(
    pl.selectors.float().round(4)
)

- Frames grouped by GT boxes: 742 frames with 1 to 24 objects, 1,044 with 25 to 49, 618 with 50 to 99, and 189 with 100 or more. All 189 densest frames come from one clip (`uav0000137_00458_v`, 2688 px wide) and contain only 92 small objects (0.5% of 20,203), so that group measures one scene and its AP_small is not meaningful.
- RT-DETRv2 has the highest mAP in the sparsest frames (0.195 against 0.178 for Faster R-CNN) and in the densest clip (0.371 against 0.367); Faster R-CNN is level with it for 25 to 99 objects.
- RetinaNet falls behind as density grows: level with Faster R-CNN in sparse frames (0.184), 0.115 below it in the densest clip. YOLO11s is weakest in sparse frames (0.131).
- AP_small: RT-DETRv2 is highest for up to 49 objects (0.107, 0.146); YOLO11s matches it for 50 to 99 (0.143).

## 7. Failure analysis

The tuned RT-DETRv2 against the tuned Faster R-CNN on all `val` frames, from their `val_predictions.parquet` (`src.evaluation.failure_analysis`). Detections are matched greedily in descending score order, class-aware, at IoU 0.5; detections at least half inside an ignored region are excluded from the counts.

### 7.1 Recall by object size

Share of GT boxes matched by any kept detection (up to 500 per frame), per COCO size bucket. This is the recall ceiling of the AP computation at IoU 0.5.

In [ ]:
ANALYSED = list(dict.fromkeys([TUNED, TUNED_FASTER_RCNN]))
val_predictions = {e: pl.read_parquet(run_dirs[e] / PREDICTIONS_FILE) for e in ANALYSED}
matches = {e: match_detections(tables, "val", val_predictions[e]) for e in ANALYSED}
recall_by_size = pl.concat(
    compute_recall_by_size(matches[e]).with_columns(pl.lit(e).alias("experiment"))
    for e in ANALYSED
)
fig = plot_grouped_bars(
    recall_by_size,
    "size_bucket",
    "recall",
    "experiment",
    ("COCO size bucket", "Recall at IoU 0.5", "Recall of all kept detections by object size (val)"),
    "{:.3f}",
)
recall_by_size.with_columns(pl.col("recall").round(3))

- RT-DETRv2 matches 86.5% of val GT boxes at IoU 0.5 with its 300 detections per frame (small 74.9%, medium 88.1%, large 96.4%), the highest recall of all families; the tuned Faster R-CNN matches 82.7% (66.4%, 84.8%, 97.0%) with up to 500.

### 7.2 Error breakdown at the best-F1 threshold

The score threshold that maximizes F1 at IoU 0.5 on `val` (ignored detections excluded), then one category per detection above it: correct, duplicate (same-class GT already matched), wrong class, poor localisation (best IoU in [0.1, 0.5)), background (best IoU < 0.1); misses are GT boxes left unmatched.

In [ ]:
thresholds = pl.DataFrame(
    [{"experiment": e, **find_best_f1_threshold(matches[e])} for e in ANALYSED]
)
error_counts = pl.concat(
    count_error_categories(matches[e], thresholds.filter(pl.col("experiment") == e)["threshold"].item())
    .with_columns(pl.lit(e).alias("experiment"))
    for e in ANALYSED
)
display(thresholds.with_columns(pl.col("threshold", "precision", "recall", "f1").round(3)))
error_counts.pivot(on="experiment", index="category", values="count")

In [ ]:
fig = plot_grouped_bars(
    error_counts,
    "category",
    "count",
    "experiment",
    ("Category", "Detections or missed GT boxes", "Error breakdown at the best-F1 threshold (val)"),
    "{:.0f}",
)

- Best-F1 threshold 0.435: precision 0.709, recall 0.677, F1 0.693, level with the tuned Faster R-CNN (0.691) at higher recall and lower precision.
- RT-DETRv2 misses fewer objects (about 34,000 against 36,000) but has more wrong-class (about 9,900 against 7,900) and poorly localised detections (about 9,700 against 8,300).
- Without NMS it still has about 680 duplicates, about 2.8 times Faster R-CNN's 240: one-to-one matching suppresses most but not all second boxes on an object.

In [ ]:
tuned_threshold = thresholds.filter(pl.col("experiment") == TUNED)["threshold"].item()
CLASS_BY_ID = dict(enumerate(CLASS_NAMES))
(
    matches[TUNED].detections.filter(
        pl.col("score") >= tuned_threshold, pl.col("category") == "wrong_class"
    )
    .group_by("best_class_id", "class_id")
    .len()
    .sort("len", "best_class_id", "class_id", descending=[True, False, False])
    .head(8)
    .select(
        pl.col("best_class_id").replace_strict(CLASS_BY_ID).alias("gt_class"),
        pl.col("class_id").replace_strict(CLASS_BY_ID).alias("predicted_class"),
        pl.col("len").alias("detections"),
    )
)

- The largest confusions are van predicted as car (about 2,360), bicycle as motor (about 1,840), motor as person (about 1,090), car as van (about 1,000), and tricycle as truck (about 540).
- These are the same pairs as in the CNN detectors, more frequent: wrong class is RT-DETRv2's largest error category after misses.

### 7.3 Example frames per category

Tuned model at its best-F1 threshold: the four highest-scoring detections of every error category and four missed GT boxes drawn at random (seed 42), at most one per clip. Each crop is centered on the example, with all GT boxes (yellow), all kept detections (blue), and shaded ignored regions; the example has a thick outline.

In [ ]:
examples = select_error_examples(
    matches[TUNED], val_images, tuned_threshold, per_category=4, seed=SEED
)
fig = plot_error_examples(
    examples,
    val_images,
    val_objects,
    val_ignored,
    val_predictions[TUNED].filter(pl.col("score") >= tuned_threshold),
    context_scale=5.0,
    min_side=96,
)
examples.join(
    val_images.select("image_id", "clip", "frame_id"), on="image_id", maintain_order="left"
).select(
    "category",
    "clip",
    "frame_id",
    pl.col("class_id").replace_strict(CLASS_BY_ID).alias("class_name"),
    pl.col("score").round(3),
    (pl.col("w") * pl.col("h")).sqrt().round(1).alias("sqrt_area_px"),
)

- Correct: vans and trucks at scores of about 0.92.
- Duplicate: cars with two overlapping boxes, in dense traffic and parking, at scores of about 0.75.
- Wrong class: confident car and truck predictions on vehicles of a neighbouring class, also at night.
- Poor localisation: boxes on trucks and cars in dense traffic that cover part of the vehicle or two vehicles.
- Background: confident vehicles without a GT box, as in the other families.
- Miss: people next to each other or in the dark, and partly hidden cars.

### 7.4 Failure cases

Four failure cases of the tuned RT-DETRv2. Counts come from the tables above; the causes are hypotheses.

1. **Class confusion** (about 9,900 wrong-class detections, the most of all families): van and car, and the two-wheelers. Each query predicts all classes at once with independent sigmoid scores, and inference ranks (query, class) pairs, so a query can also contribute a second-best class.
2. **Duplicates without NMS** (about 680): one-to-one matching trains second queries on an object as background, but in dense traffic two queries sometimes keep confident boxes on one car.
3. **Small objects**: 25% of small boxes are never detected, against 3.6% of large ones, although RT-DETRv2 is the best family on small objects.
4. **Label issues shared with the other families**: loose GT boxes behind poor localisation and unlabelled vehicles behind background detections (`02_two_stage.ipynb` §7.4).

## 8. Comparison

RT-DETRv2 against the tuned model of every family by class and by size; the last table lists the baseline rows of all families.

In [ ]:
EXTRA_COMPARED = ["retinanet-baseline", "yolo-baseline"]

In [ ]:
COMPARED = list(dict.fromkeys([*ANALYSED, *EXTRA_COMPARED]))
per_class_ap = pl.DataFrame(
    {
        "class_name": list(CLASS_NAMES),
        **{
            e: [round(run_metrics[e]["evaluation"]["per_class_ap"][name], 4) for name in CLASS_NAMES]
            for e in COMPARED
        },
    }
)
per_class_ap

- RT-DETRv2 has the best AP of the four tuned models on 6 of 8 classes: person, car, van, tricycle, bus, and motor.
- The tuned Faster R-CNN is better on bicycle (0.296 against 0.274) and truck (0.317 against 0.245).

In [ ]:
compare_with_reference(
    results.filter(pl.col("experiment").is_in(COMPARED)),
    TUNED_FASTER_RCNN,
    ["map", "map50", "map75", "ap_small", "ap_medium", "ap_large", "ms_per_image"],
).with_columns(pl.selectors.float().round(4))

- Against the tuned Faster R-CNN, RT-DETRv2 gains 0.0143 mAP: AP75 +0.0406, AP_small +0.0131, AP_large +0.0330, but AP50 −0.0231 and AP_medium −0.0205. Its boxes are more precise when correct, while Faster R-CNN finds more medium objects at IoU 0.5.
- It is 7.5 ms per frame faster (53.2 against 60.8 ms).

In [ ]:
results.filter(pl.col("experiment").str.ends_with("-baseline")).select(
    "experiment", "epochs", "gpu_hours", "map", "map50", "ap_small", "ap_medium", "ap_large",
    "ms_per_image", "public_lb",
).with_columns(pl.col("gpu_hours", "ms_per_image").round(2))

RT-DETRv2 has the best baseline on full val (0.3209) and on the public LB (0.2563); the order of the four baselines on the LB is RT-DETRv2, Faster R-CNN, YOLO11s, RetinaNet.

## 9. Conclusions

**Results**

- **Baseline.** COCO-pretrained RT-DETRv2 R50vd reaches 0.3209 mAP, 0.6027 AP50, and 0.1334 AP_small on full val at 53.2 ms/image after 1.43 GPU hours, the best of all families on val and on the public LB (0.2563) (§4).
- **Tuning.** Scaling both learning rates to the effective batch (`lr_scaled`, +0.0096) and doubling the denoising queries (`denoising_groups`, +0.0065) raise mAP at seed 42, mostly through bus and truck; the seed repeat of `lr_scaled` falls back to +0.0002. The seed spread of 0.0094 mAP is ten times that of Faster R-CNN, so the baseline stays the tuned model (§5).
- **Matching and no NMS.** One-to-one matching lets RT-DETRv2 output a fixed 300 detections without NMS; it still leaves about 680 duplicates at the best-F1 threshold, 2.8 times Faster R-CNN's 240 (§6.1, §7.2).
- **Convergence.** Monitoring mAP is 0.320 after the first epoch and peaks after 0.40 of 1.42 GPU hours, then declines while the training losses keep falling. The model converged on val within the budget; it did not converge in training loss (§6.4).
- **Density and size.** RT-DETRv2 leads in the sparsest frames and in the densest clip, is level with Faster R-CNN in between, and has the highest recall of all families (86.5%, small 74.9%) (§6.5, §7.1). It gains AP75, AP_small, and AP_large over the tuned Faster R-CNN but loses AP50 and AP_medium (§8).

**Hypotheses (not tested here)**

- The early peak with falling training losses is overfitting on unaugmented frames from 56 clips; augmentation and more varied frames should help more than tuning the learning rate.
- Class confusion is RT-DETRv2's main error after misses; an ensemble with Faster R-CNN, which confuses classes less and finds more medium objects at IoU 0.5, should combine their strengths (`docs/pipeline.md`, M2).